# 1. Install the required packages



In [1]:
%pip -q install scikit-learn pandas joblib

# 2. Load and split the data

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer(as_frame=True)

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class labels:", dict(enumerate(dataset.target_names)))

Training shape: (455, 30)
Test shape: (114, 30)
Class labels: {0: np.str_('malignant'), 1: np.str_('benign')}


# 3 Train the Model

In [3]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("Training complete.")

Training complete.


# 4. Evaluate the model

In [4]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, predictions)),
    "roc_auc": float(roc_auc_score(y_test, probabilities)),
}

print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"ROC-AUC:  {metrics['roc_auc']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=dataset.target_names,
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.9561
ROC-AUC:  0.9934

Classification report:
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Confusion matrix:
[[39  3]
 [ 2 70]]


# 5. Save and download the trained model

In [5]:
import json
import joblib
import sklearn

artifact = {
    "model": model,
    "feature_names": X.columns.tolist(),
    "target_names": dataset.target_names.tolist(),
    "sklearn_version": sklearn.__version__,
}

joblib.dump(artifact, "random_forest_model.joblib")

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Model and metrics saved.")

Model and metrics saved.


In [6]:
from google.colab import files

files.download("random_forest_model.joblib")
files.download("metrics.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 6. Make a prediction

In [7]:
sample = X_test.iloc[[0]]

predicted_class = int(model.predict(sample)[0])
class_probabilities = model.predict_proba(sample)[0]

print("Predicted label:", dataset.target_names[predicted_class])
print("Actual label:", dataset.target_names[int(y_test.iloc[0])])

print("\nClass probabilities:")
for name, probability in zip(
    dataset.target_names,
    class_probabilities,
):
    print(f"{name}: {probability:.4f}")

Predicted label: malignant
Actual label: malignant

Class probabilities:
malignant: 0.9996
benign: 0.0004
